# A/B Testing - Statistical Significance Analysis

## Project Description
Аналіз статистичної значущості A/B тестів для 4 конверсійних метрик
з використанням двовибіркового z-test (statsmodels).

## Links
- 📊 [Tableau Dashboard](https://public.tableau.com/app/profile/anna.savchuk6598/viz/Signification/Significance?publish=yes)
- 📁 [Results CSV](https://drive.google.com/file/d/1dWZTuwdvLWY5vkwTkuplraOESM_HfZkl/view?usp=sharing)

In [ ]:
import pandas as pd
import statsmodels.api as sm
from google.colab import drive

drive.mount("/content/drive")
FILE_PATH = '/content/drive/MyDrive/Portfolio/test_info.csv'

Mounted at /content/drive


In [ ]:

df = pd.read_csv(FILE_PATH)
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head()
METRICS = ['add_payment_info', 'add_shipping_info', 'begin_checkout', 'newaccount']
ALPHA   = 0.05

# test_group: 1 = control (A), 2 = test (B)
GROUP_A = 1
GROUP_B = 2

Shape: (800996, 9)
Columns: ['date', 'country', 'device', 'continent', 'channel', 'test', 'test_group', 'event_name', 'value']


In [ ]:
def calc_significance(data, segment_type, segment_value, test_id='All'):

    pivot = pd.pivot_table(
        data,
        values='value',
        index='event_name',
        columns='test_group',
        aggfunc='sum'
    ).fillna(0)  # якщо якась група відсутня — заповнюємо 0

    # Пропускаємо якщо немає сесій або однієї з груп
    if 'session' not in pivot.index:
        return pd.DataFrame()
    if GROUP_A not in pivot.columns or GROUP_B not in pivot.columns:
        return pd.DataFrame()

    n_A = pivot.loc['session', GROUP_A]
    n_B = pivot.loc['session', GROUP_B]

    # Пропускаємо якщо сесій немає
    if n_A == 0 or n_B == 0:
        return pd.DataFrame()

    rows = []
    for m in METRICS:
        if m not in pivot.index:
            continue

        conv_A = pivot.loc[m, GROUP_A]
        conv_B = pivot.loc[m, GROUP_B]

        cr_A = conv_A / n_A
        cr_B = conv_B / n_B
        metric_change = ((cr_B / cr_A) - 1) * 100 if cr_A else 0

        z_stat, p_value = sm.stats.proportions_ztest(
            [conv_A, conv_B],
            [n_A,    n_B]
        )

        rows.append({
            'test_id'                : test_id, # Доданий ідентифікатор тесту
            'segment_type'           : segment_type,
            'segment_value'          : segment_value,
            'metric'                 : f'{m}/session',
            'numerator_event'        : m,
            'denominator_event'      : 'session',
            'numerator_control'      : conv_A,
            'denominator_control'    : n_A,
            'conversion_rate_control': round(cr_A, 6),
            'numerator_test'         : conv_B,
            'denominator_test'       : n_B,
            'conversion_rate_test'   : round(cr_B, 6),
            'metric_change_pct'      : round(metric_change, 2),
            'z_stat'                 : round(z_stat, 6),
            'p_value'                : round(p_value, 6),
            'significant'            : p_value < ALPHA
        })

    return pd.DataFrame(rows)

In [ ]:
df_result = pd.DataFrame()

# 1. Overall
df_result = pd.concat([df_result, calc_significance(df, 'overall', 'all', 'All Tests')])

# Проходимося по кожному тесту окремо (Головний цикл)
for test_num in sorted(df['test'].unique()):

    # Фільтруємо датафрейм ТІЛЬКИ для поточного тесту
    test_subset = df[df['test'] == test_num]
    t_id = f'Test {test_num}'

    # 2. By test (Загальний результат саме для цього тесту)
    df_result = pd.concat([df_result, calc_significance(test_subset, 'test', 'all', t_id)])

    # 3. By country (В межах поточного тесту)
    # Використовуємо .dropna() на випадок порожніх значень, щоб sorted() не видав помилку
    for country in sorted(test_subset['country'].dropna().unique()):
        subset = test_subset[test_subset['country'] == country]
        df_result = pd.concat([df_result, calc_significance(subset, 'country', country, t_id)])

    # 4. By device (В межах поточного тесту)
    for device in sorted(test_subset['device'].dropna().unique()):
        subset = test_subset[test_subset['device'] == device]
        df_result = pd.concat([df_result, calc_significance(subset, 'device', device, t_id)])

    # 5. By continent (В межах поточного тесту)
    for continent in sorted(test_subset['continent'].dropna().unique()):
        subset = test_subset[test_subset['continent'] == continent]
        df_result = pd.concat([df_result, calc_significance(subset, 'continent', continent, t_id)])

    # 6. By channel (В межах поточного тесту)
    for channel in sorted(test_subset['channel'].dropna().unique()):
        subset = test_subset[test_subset['channel'] == channel]
        df_result = pd.concat([df_result, calc_significance(subset, 'channel', channel, t_id)])

df_result = df_result.reset_index(drop=True)

print('Total rows:', len(df_result))
print('Significant:', df_result['significant'].sum())
df_result.head(25)

/usr/local/lib/python3.12/dist-packages/statsmodels/stats/proportion.py:1024: RuntimeWarning: invalid value encountered in sqrt
  std_diff = np.sqrt(var_)


Total rows: 1815
Significant: 377


,test_id,segment_type,segment_value,metric,numerator_event,denominator_event,numerator_control,denominator_control,conversion_rate_control,numerator_test,denominator_test,conversion_rate_test,metric_change_pct,z_stat,p_value,significant
0,All Tests,overall,all,add_payment_info/session,add_payment_info,session,11686.0,271125.0,0.043102,11936.0,271017.0,0.044042,2.18,-1.694549,0.090161,False
1,All Tests,overall,all,add_shipping_info/session,add_shipping_info,session,16940.0,271125.0,0.062480,16875.0,271017.0,0.062265,-0.34,0.327211,0.743508,False
2,All Tests,overall,all,begin_checkout/session,begin_checkout,session,30133.0,271125.0,0.111141,29865.0,271017.0,0.110196,-0.85,1.108462,0.267662,False
3,All Tests,overall,all,newaccount/session,newaccount,session,22828.0,271125.0,0.084197,22374.0,271017.0,0.082556,-1.95,2.186156,0.028804,True
4,Test 1,test,all,add_payment_info/session,add_payment_info,session,1988.0,45362.0,0.043825,2229.0,45193.0,0.049322,12.54,-3.924884,0.000087,True
5,Test 1,test,all,add_shipping_info/session,add_shipping_info,session,3034.0,45362.0,0.066884,3221.0,45193.0,0.071272,6.56,-2.603571,0.009226,True
6,Test 1,test,all,begin_checkout/session,begin_checkout,session,3784.0,45362.0,0.083418,4021.0,45193.0,0.088974,6.66,-2.978783,0.002894,True
7,Test 1,test,all,newaccount/session,newaccount,session,3823.0,45362.0,0.084278,3681.0,45193.0,0.081451,-3.35,1.542883,0.122859,False
8,Test 1,country,(not set),add_payment_info/session,add_payment_info,session,16.0,369.0,0.043360,19.0,373.0,0.050938,17.48,-0.486827,0.626381,False
9,Test 1,country,(not set),add_shipping_info/session,add_shipping_info,session,23.0,369.0,0.062331,26.0,373.0,0.069705,11.83,-0.404423,0.685902,False


In [ ]:
# ── ВИСНОВКИ  ────────────────────────────────────────────────────────
sig = df_result[df_result['significant']].copy()


cols_to_show = ['test_id', 'segment_value', 'metric', 'conversion_rate_control', 'conversion_rate_test', 'metric_change_pct', 'p_value']

# По тестах (Загальні результати)
print('=== ЗНАЧУЩІ РЕЗУЛЬТАТИ ПО ТЕСТАХ ===')
display(sig[sig['segment_type'] == 'test'][cols_to_show])

# По країнах
print('\n=== ЗНАЧУЩІ РЕЗУЛЬТАТИ ПО КРАЇНАХ ===')
display(sig[sig['segment_type'] == 'country'][cols_to_show].sort_values(by=['test_id', 'metric_change_pct']))

# По девайсах
print('\n=== ЗНАЧУЩІ РЕЗУЛЬТАТИ ПО ДЕВАЙСАХ ===')
display(sig[sig['segment_type'] == 'device'][cols_to_show].sort_values(by=['test_id', 'metric_change_pct']))

# По континентах
print('\n=== ЗНАЧУЩІ РЕЗУЛЬТАТИ ПО КОНТИНЕНТАХ ===')
display(sig[sig['segment_type'] == 'continent'][cols_to_show].sort_values(by=['test_id', 'metric_change_pct']))

# По каналах
print('\n=== ЗНАЧУЩІ РЕЗУЛЬТАТИ ПО КАНАЛАХ ===')
display(sig[sig['segment_type'] == 'channel'][cols_to_show].sort_values(by=['test_id', 'metric_change_pct']))

=== ЗНАЧУЩІ РЕЗУЛЬТАТИ ПО ТЕСТАХ ===


,test_id,segment_value,metric,conversion_rate_control,conversion_rate_test,metric_change_pct,p_value
4,Test 1,all,add_payment_info/session,0.043825,0.049322,12.54,0.000087
5,Test 1,all,add_shipping_info/session,0.066884,0.071272,6.56,0.009226
6,Test 1,all,begin_checkout/session,0.083418,0.088974,6.66,0.002894
895,Test 3,all,begin_checkout/session,0.136080,0.131518,-3.35,0.012026
1343,Test 4,all,begin_checkout/session,0.119482,0.116672,-2.35,0.045934
1344,Test 4,all,newaccount/session,0.085498,0.082622,-3.36,0.017527



=== ЗНАЧУЩІ РЕЗУЛЬТАТИ ПО КРАЇНАХ ===


,test_id,segment_value,metric,conversion_rate_control,conversion_rate_test,metric_change_pct,p_value
13,Test 1,Albania,add_shipping_info/session,0.222222,0.000000,-100.00,0.049311
14,Test 1,Albania,begin_checkout/session,0.333333,0.000000,-100.00,0.013823
83,Test 1,Costa Rica,add_payment_info/session,0.312500,0.000000,-100.00,0.004088
126,Test 1,Georgia,add_payment_info/session,0.157895,0.000000,-100.00,0.035971
127,Test 1,Georgia,add_shipping_info/session,0.210526,0.000000,-100.00,0.014244
...,...,...,...,...,...,...,...
1725,Test 4,Tunisia,begin_checkout/session,0.062500,0.305085,388.14,0.001676
1666,Test 4,Romania,add_payment_info/session,0.003448,0.022140,542.07,0.046257
1381,Test 4,Bangladesh,add_payment_info/session,0.009901,0.064516,551.61,0.003559
1547,Test 4,Kazakhstan,begin_checkout/session,0.030303,0.218182,620.00,0.001295



=== ЗНАЧУЩІ РЕЗУЛЬТАТИ ПО ДЕВАЙСАХ ===


,test_id,segment_value,metric,conversion_rate_control,conversion_rate_test,metric_change_pct,p_value
400,Test 1,tablet,add_payment_info/session,0.048048,0.030723,-36.06,0.045868
402,Test 1,tablet,begin_checkout/session,0.083083,0.055500,-33.20,0.014907
392,Test 1,desktop,add_payment_info/session,0.042695,0.047545,11.36,0.007210
393,Test 1,desktop,add_shipping_info/session,0.064647,0.072529,12.19,0.000336
394,Test 1,desktop,begin_checkout/session,0.079646,0.091002,14.26,0.000003
396,Test 1,mobile,add_payment_info/session,0.045262,0.053020,17.14,0.000701
839,Test 2,desktop,begin_checkout/session,0.081466,0.087985,8.00,0.004507
1295,Test 3,tablet,begin_checkout/session,0.116461,0.085823,-26.31,0.004578
1291,Test 3,mobile,begin_checkout/session,0.136294,0.129891,-4.70,0.027319
1767,Test 4,tablet,add_payment_info/session,0.044977,0.022881,-49.13,0.000027



=== ЗНАЧУЩІ РЕЗУЛЬТАТИ ПО КОНТИНЕНТАХ ===


,test_id,segment_value,metric,conversion_rate_control,conversion_rate_test,metric_change_pct,p_value
410,Test 1,Africa,begin_checkout/session,0.109312,0.069620,-36.31,0.030894
415,Test 1,Americas,newaccount/session,0.086036,0.079575,-7.51,0.008827
421,Test 1,Europe,add_shipping_info/session,0.062323,0.071946,15.44,0.012445
422,Test 1,Europe,begin_checkout/session,0.076133,0.088330,16.02,0.003907
420,Test 1,Europe,add_payment_info/session,0.038244,0.053663,40.32,0.000002
426,Test 1,Oceania,begin_checkout/session,0.070727,0.113131,59.96,0.019894
405,Test 1,(not set),add_shipping_info/session,0.041237,0.130000,215.25,0.026545
406,Test 1,(not set),begin_checkout/session,0.041237,0.220000,433.50,0.000211
853,Test 2,Africa,add_payment_info/session,0.056537,0.022449,-60.29,0.005188
855,Test 2,Africa,begin_checkout/session,0.113074,0.051020,-54.88,0.000295



=== ЗНАЧУЩІ РЕЗУЛЬТАТИ ПО КАНАЛАХ ===


,test_id,segment_value,metric,conversion_rate_control,conversion_rate_test,metric_change_pct,p_value
432,Test 1,Organic Search,add_payment_info/session,0.040829,0.032883,-19.46,0.000191
447,Test 1,Undefined,newaccount/session,0.090228,0.072711,-19.41,0.008584
433,Test 1,Organic Search,add_shipping_info/session,0.065136,0.058985,-9.44,0.024127
434,Test 1,Organic Search,begin_checkout/session,0.079681,0.073188,-8.15,0.030622
429,Test 1,Direct,add_shipping_info/session,0.062108,0.069105,11.27,0.040295
436,Test 1,Paid Search,add_payment_info/session,0.038210,0.043831,14.71,0.029452
430,Test 1,Direct,begin_checkout/session,0.076981,0.088312,14.72,0.002821
428,Test 1,Direct,add_payment_info/session,0.036666,0.049802,35.83,0.000003
444,Test 1,Undefined,add_payment_info/session,0.080635,0.122461,51.87,0.000000
445,Test 1,Undefined,add_shipping_info/session,0.083633,0.128349,53.47,0.000000


## Висновки

### Що ми перевіряємо
У кожному A/B тесті користувачі розділені на дві групи:
- **Control (група 1)** — бачить стару версію
- **Test (група 2)** — бачить нову версію

Ми перевіряємо чи змінилась конверсія по 4 метриках:
- `add_payment_info/session` — частка сесій де додали платіжні дані
- `add_shipping_info/session` — частка сесій де додали адресу доставки
- `begin_checkout/session` — частка сесій де розпочали оформлення
- `newaccount/session` — частка сесій де створили акаунт


Якщо результат **значущий** (p_value < 0.05) — різниця між групами
реальна і не є випадковістю. Можна приймати рішення на основі цих даних.

Якщо результат **не значущий** — різниця могла виникнути випадково.
Недостатньо підстав щоб стверджувати що нова версія краща або гірша.

---


##  Загальний підсумок (Executive Summary)
* **Тест 1** — єдиний однозначно успішний варіант, який суттєво покращив проходження користувачів через воронку. Рекомендується до масштабування, але з обов'язковим технічним аудитом версії для планшетів.
* **Тест 2** — на загальному рівні не показав статистично значущих змін, проте глибокий аналіз сегментів виявив критичну проблему з органічним трафіком та аномалії в географії.
* **Тести 3 та 4** — показали негативний вплив на ключові конверсійні дії (початок оформлення замовлення та реєстрацію). Впровадження цих версій завдасть шкоди бізнесу.

---

##  Детальний аналіз по кожному тесту

###  Тест 1 (Успішний)
* **Загальний вплив:** Нова версія значно підвищила ефективність воронки. Додавання платіжних даних зросло на **+12.54%**, додавання адреси доставки — на **+6.56%**, а перехід до чекауту — на **+6.66%**.
* **Драйвери росту:** * Основний внесок зробив ринок **Європи**, де метрика `add_payment_info` злетіла на **+40.32%**, а чекаут — на **+16.02%**.
  * З точки зору девайсів успіх забезпечили користувачі **Desktop** (+14.26% у чекауті) та **Mobile** (+17.14% у додаванні платежу).
  * Користувачі з каналу **Direct** показали сильний приріст конверсії (+14-35%).
* **Критичні зони (Red Flags):**  на **Tablet** — конверсія впала на понад **-33...-36%**. Також спостерігається просадка в каналі **Organic Search** (-8...-19%).

###  Тест 2 (Нейтральний загалом, аномальний у сегментах)
* **Загальний вплив:** На рівні всього трафіку статистично значущих змін не зафіксовано.
* **Інсайти по сегментах:**
  * **Обвал органіки та Африки:** Користувачі з **Organic Search** стали гірше конвертуватися (-9...-14%), а ринок Африки показав катастрофічне падіння на **-51...-60%**.
  * **Аномальний трафік:** Канал **Undefined** (невизначене джерело) показав різкий стрибок на **+25-31%**. Це вагомий сигнал про можливий наплив бот-трафіку в тестову групу, що могло викривити загальні результати тесту. Потрібен технічний аудит логів.

###  Тест 3 (Негативний)
* **Загальний вплив:** Значуще зниження переходу до оформлення замовлення (`begin_checkout`) на **-3.35%**.
* **Інсайти по сегментах:** Падіння здебільшого зумовлене погіршенням показників на мобільних пристроях (**Mobile** -4.7%) та планшетах (**Tablet** -26.31%). Також негативно відреагував ринок Америки (-4.89%) та органічний пошук.

###  Тест 4 (Найменш успішний)
* **Загальний вплив:** Нова версія погіршила одразу дві ключові метрики: `begin_checkout` впав на **-2.35%**, а створення нових акаунтів (`newaccount`) — на **-3.36%**.
* **Інсайти по сегментах:**
  * Повний обвал на **Tablet** (до **-49.13%** на етапі додавання платежу) та погіршення на **Desktop** (всі метрики впали на 5-7%).
  * Трафік із **Social Search** відреагував вкрай негативно (-9...-20%).
* *Цікава деталь:* незважаючи на загальний провал, метрика чекауту на **Mobile** дещо зросла (+4.29%).

---

## 💡 Стратегічні рекомендації
1. **Зупинити розгортання Тестів 3 та 4.** Їхні поточні версії шкодять конверсії.
2. **Підготувати Тест 1 до релізу**, але **тільки** для десктопної та мобільної версій. Версію для планшетів необхідно повернути на доопрацювання дизайнерам/фронтендерам, оскільки вона блокує користувачів.
3. **Дослідити проблему Organic Search.** У всіх тестах (особливо 1, 2 та 4) спостерігається стабільне падіння конверсії з органічного пошуку. Можливо, нові версії сторінок вантажаться довше або порушують звичний патерн поведінки SEO-трафіку.
4. **Перевірити якість даних.** Аномальний ріст у сегменті `Undefined` (Тести 1, 2, 3) вимагає перевірки налаштувань аналітики Google Analytics/системи трекінгу або наявності захисту від ботів.

In [ ]:
# ── SAVE ───────────────────────────────────────────────────────────────────
OUT = '/content/drive/MyDrive/ab_testing_results.csv'
df_result.to_csv('/content/drive/MyDrive/ab_testing_results.csv',
                 index=False,
                 sep=';')

#Dashboard Tableau:
https://public.tableau.com/app/profile/anna.savchuk6598/viz/Signification/Significance?publish=yes